# Fine-Tuning de Transformers (BETO y RoBERTa) para Clasificación de Emociones

Curso: Minería de Datos  |  Grupo: 04

Proyecto: Clasificación automática de emociones en publicaciones de jóvenes peruanos

## 1. Verificación de GPU

Asegurarse de que Google Colab esté usando una GPU (Runtime → Cambiar tipo de entorno → T4 GPU).

In [1]:
import torch
import sys

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo detectado: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("[!] NO se detectó GPU. El entrenamiento será muy lento. Activa GPU en Runtime > Cambiar tipo de entorno.")

Dispositivo detectado: cuda
GPU: Tesla T4
Memoria: 15.6 GB


## 2. Instalación de dependencias

In [2]:
!pip install transformers datasets evaluate accelerate scikit-learn pandas matplotlib seaborn openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, classification_report
)
from sklearn.preprocessing import LabelBinarizer
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments,
    EarlyStoppingCallback
)
from accelerate import Accelerator

print("Todas las librerías se importaron correctamente.")

Todas las librerías se importaron correctamente.


## 3. Carga y unificación de datasets

Se cargan los 3 datasets preprocesados (YouTube parte 1, YouTube parte 2, TikTok) y se unifican en un solo DataFrame, replicando la lógica de `app.py`.

In [4]:
# --- Configurar rutas de acuerdo al entorno ---
import os
import sys

if os.path.exists('/content'):
    # ENTORNOS NUBE (Google Colab)
    print("🌍 Entorno Nube (Colab) Detectado.")
    print("Montando Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    BASE_DIR = '/content'
    # Ruta apuntando a la carpeta de Google Drive especificada por el usuario
    DATOS_DIR = '/content/drive/MyDrive/Proyecto-Mineria-G4/prueba'
else:
    # ENTORNOS LOCALES (VS Code, Jupyter Local)
    print("💻 Entorno Local Detectado.")
    try:
        current_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        current_dir = os.getcwd()

    BASE_DIR = current_dir
    while BASE_DIR != os.path.dirname(BASE_DIR):
        if 'modelado' in os.listdir(BASE_DIR) and os.path.isdir(os.path.join(BASE_DIR, 'modelado')):
            break
        BASE_DIR = os.path.dirname(BASE_DIR)
        
    DATOS_DIR = os.path.join(BASE_DIR, 'modelado', 'datos')

if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

RESULTADOS_DIR = os.path.join(BASE_DIR, 'resultados_transformers')
os.makedirs(RESULTADOS_DIR, exist_ok=True)

print(f"Directorio base detectado: {BASE_DIR}")
print(f"Directorio de datos: {DATOS_DIR}")


Mounted at /content/drive
Directorio base detectado: /content
Directorio de datos: /content/drive/MyDrive/Proyecto-Mineria-G4/prueba


In [15]:
import glob
import pandas as pd

archivos = [
    os.path.join(DATOS_DIR, 'dataset_procesado_tiktok_parte01.csv'),
    os.path.join(DATOS_DIR, 'dataset_procesado_youtube_parte01.csv'),
    os.path.join(DATOS_DIR, 'dataset_procesado_youtube_parte02.csv')
]

dfs = []
for f in archivos:
    if os.path.exists(f):
        print(f"Cargando {os.path.basename(f)}...")
        temp_df = pd.read_csv(f, sep=';', encoding='utf-8')
        temp_df.columns = temp_df.columns.str.strip().str.replace('"', '').str.lower()
        dfs.append(temp_df)

if not dfs:
    raise ValueError("No se encontraron los datasets en modelado/datos/")

df = pd.concat(dfs, ignore_index=True)
df['emocion'] = df['emocion'].astype(str).str.strip().str.replace('"', '').str.capitalize()
df['emocion'] = df['emocion'].replace({'Alegría': 'Alegria'})

# Filtrar 6 emociones maestras
emociones_validas = ['Sorpresa', 'Miedo', 'Alegria', 'Tristeza']
df = df[df['emocion'].isin(emociones_validas)]
# Usamos la nueva columna del dataset consolidado
df = df.dropna(subset=['content_transformers', 'emocion'])

print(f"\nDataset unificado: {len(df)} registros")
print(f"Distribución de emociones:\n{df['emocion'].value_counts()}")


Cargando dataset_procesado_tiktok_parte01.csv...
Cargando dataset_procesado_youtube_parte01.csv...
Cargando dataset_procesado_youtube_parte02.csv...

Dataset unificado: 9065 registros
Distribución de emociones:
emocion
Tristeza    2684
Alegria     2329
Sorpresa    2048
Miedo       2004
Name: count, dtype: int64


## 4. División entrenamiento/prueba y preparación de datos

Se usan los lemas generados en el preprocesamiento como texto de entrada. División 70/30 estratificada (misma semilla que los otros pipelines: random_state=42).

In [16]:
# Separar características y etiquetas
# CRUCIAL: Utilizar la columna content_transformers en lugar de la original
textos = df['content_transformers'].astype(str).tolist()
etiquetas = df['emocion'].tolist()
clases = sorted(df['emocion'].unique().tolist())
print(f"Clases: {clases}")

# División estratificada (70/30)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    textos, etiquetas, test_size=0.3, random_state=42, stratify=etiquetas
)

print(f"\nEntrenamiento: {len(train_texts)} registros")
print(f"Prueba: {len(test_texts)} registros")

# Mapa de etiquetas a índices
label2id = {e: i for i, e in enumerate(clases)}
id2label = {i: e for i, e in enumerate(clases)}
print(f"\nMapa de etiquetas: {label2id}")

train_labels_ids = [label2id[l] for l in train_labels]
test_labels_ids = [label2id[l] for l in test_labels]

Clases: ['Alegria', 'Miedo', 'Sorpresa', 'Tristeza']

Entrenamiento: 6345 registros
Prueba: 2720 registros

Mapa de etiquetas: {'Alegria': 0, 'Miedo': 1, 'Sorpresa': 2, 'Tristeza': 3}


### 4.1 Pesos por clase (para manejar desbalance)

Se calculan pesos inversamente proporcionales a la frecuencia de cada clase, para que el modelo preste más atención a las clases minoritarias (Ira, Asco).

In [17]:
from sklearn.utils.class_weight import compute_class_weight

classes_array = np.array(clases)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes_array,
    y=np.array(train_labels)
)
class_weights_dict = {clases[i]: class_weights[i] for i in range(len(clases))}
pesos_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Pesos por clase (balanced):")
for c, w in class_weights_dict.items():
    print(f"   {c}: {w:.4f}")

Pesos por clase (balanced):
   Alegria: 0.9732
   Miedo: 1.1306
   Sorpresa: 1.1069
   Tristeza: 0.8442


### 4.2 Clase Dataset de PyTorch

In [18]:
class EmotionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

## 5. Funciones de evaluación

Se definen las funciones para calcular métricas y generar gráficos (matriz de confusión y curva ROC multiclase).

In [19]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision_macro': precision_score(labels, predictions, average='macro', zero_division=0),
        'recall_macro': recall_score(labels, predictions, average='macro', zero_division=0),
        'f1_macro': f1_score(labels, predictions, average='macro', zero_division=0)
    }


def generar_matriz_confusion(y_true, y_pred, clases, titulo, filename, ruta):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(clases)))
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=clases, yticklabels=clases)
    plt.title(f'Matriz de Confusión - {titulo}')
    plt.xlabel('Predicción')
    plt.ylabel('Real')
    plt.xticks(rotation=45)
    plt.tight_layout()
    ruta_completa = os.path.join(ruta, filename)
    plt.savefig(ruta_completa, dpi=300)
    plt.close()
    return ruta_completa


def generar_curva_roc(y_true, y_scores, clases, titulo, filename, ruta):
    lb = LabelBinarizer()
    y_true_bin = lb.fit_transform(y_true)

    plt.figure(figsize=(10, 8))

    fpr_dict = {}
    tpr_dict = {}
    roc_auc = {}

    for i, clase in enumerate(clases):
        fpr_dict[i], tpr_dict[i], _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
        roc_auc[i] = auc(fpr_dict[i], tpr_dict[i])
        plt.plot(fpr_dict[i], tpr_dict[i], lw=2,
                 label=f'{clase} (AUC = {roc_auc[i]:.3f})')

    # Macro-average
    all_fpr = np.unique(np.concatenate([fpr_dict[i] for i in range(len(clases))]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(len(clases)):
        mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
    mean_tpr /= len(clases)
    macro_auc = auc(all_fpr, mean_tpr)
    plt.plot(all_fpr, mean_tpr, 'k--', lw=3,
             label=f'Macro-promedio (AUC = {macro_auc:.3f})')

    plt.plot([0, 1], [0, 1], 'gray', linestyle=':', lw=1)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Tasa de Falsos Positivos (FPR)')
    plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
    plt.title(f'Curva ROC - {titulo}')
    plt.legend(loc='lower right', fontsize=9)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    ruta_completa = os.path.join(ruta, filename)
    plt.savefig(ruta_completa, dpi=300)
    plt.close()
    return ruta_completa, roc_auc, macro_auc


def reporte_por_clase(y_true, y_pred, clases):
    report = classification_report(y_true, y_pred, labels=range(len(clases)),
                                   target_names=clases, zero_division=0, output_dict=True)
    return report


def guardar_reporte_markdown(filename, ruta, nombre_modelo, metricas, cm_path, roc_path,
                             roc_auc_dict, macro_auc, reporte_clases, y_true, y_pred, clases):
    ruta_completa = os.path.join(ruta, filename)
    cm_rel = cm_path.replace('\\', '/')
    roc_rel = roc_path.replace('\\', '/')

    with open(ruta_completa, 'w', encoding='utf-8') as f:
        f.write(f"# Reporte de Evaluación - {nombre_modelo}\n\n")
        f.write(f"Resultados de métricas de rendimiento para el modelo {nombre_modelo}.\n\n")

        f.write("## Métricas Globales\n\n")
        f.write(f"- **Accuracy:** {metricas['accuracy']:.4f}\n")
        f.write(f"- **Precision (macro):** {metricas['precision_macro']:.4f}\n")
        f.write(f"- **Recall (macro):** {metricas['recall_macro']:.4f}\n")
        f.write(f"- **F1-Score (macro):** {metricas['f1_macro']:.4f}\n\n")

        f.write("## AUC (Área Bajo la Curva ROC)\n\n")
        for i, clase in enumerate(clases):
            f.write(f"- **{clase}:** {roc_auc_dict[i]:.4f}\n")
        f.write(f"- **Macro-promedio:** {macro_auc:.4f}\n\n")

        f.write("## Reporte por Clase\n\n")
        f.write("| Clase | Precisión | Recall | F1-Score | Soporte |\n")
        f.write("|-------|-----------|--------|----------|---------|\n")
        for clase in clases:
            r = reporte_clases[clase]
            f.write(f"| {clase} | {r['precision']:.4f} | {r['recall']:.4f} | {r['f1-score']:.4f} | {int(r['support'])} |\n")
        f.write("\n")

        f.write("## Matriz de Confusión\n\n")
        f.write(f"![Matriz de Confusión {nombre_modelo}]({cm_rel})\n\n")

        f.write("## Curva ROC\n\n")
        f.write(f"![Curva ROC {nombre_modelo}]({roc_rel})\n\n")

        f.write("## Classification Report (detallado)\n\n")
        f.write("```text\n")
        f.write(classification_report(y_true, y_pred, labels=range(len(clases)),
                                      target_names=clases, zero_division=0))
        f.write("```\n")

    print(f"Reporte guardado: {ruta_completa}")
    return ruta_completa

---
## 6. Entrenamiento: RoBERTuito (Modelo Especializado en Emociones)

Modelo: [`pysentimiento/robertuito-emotion-analysis`](https://huggingface.co/pysentimiento/robertuito-emotion-analysis)

Pre-entrenado en tweets en español, ideal para textos de redes sociales (informales, cortos).

In [20]:
MODELO_BETO = 'pysentimiento/robertuito-emotion-analysis'
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 100

print(f"Configuración del Modelo Base:")
print(f"   Modelo: {MODELO_BETO}")
print(f"   Max length: {MAX_LENGTH}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Weight decay: {WEIGHT_DECAY}")
print(f"   Warmup steps: {WARMUP_STEPS}")

Configuración del Modelo Base:
   Modelo: pysentimiento/robertuito-emotion-analysis
   Max length: 128
   Batch size: 16
   Epochs: 3
   Learning rate: 2e-05
   Weight decay: 0.01
   Warmup steps: 100


In [21]:
print("Cargando tokenizer...")
tokenizer_beto = AutoTokenizer.from_pretrained(MODELO_BETO)

print("Tokenizando textos de entrenamiento...")
train_encodings_beto = tokenizer_beto(
    train_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors=None
)

print("Tokenizando textos de prueba...")
test_encodings_beto = tokenizer_beto(
    test_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors=None
)

train_dataset_beto = EmotionDataset(train_encodings_beto, train_labels_ids)
test_dataset_beto = EmotionDataset(test_encodings_beto, test_labels_ids)

print(f"Train dataset: {len(train_dataset_beto)} muestras")
print(f"Test dataset: {len(test_dataset_beto)} muestras")

Cargando tokenizer...
Tokenizando textos de entrenamiento...
Tokenizando textos de prueba...
Train dataset: 6345 muestras
Test dataset: 2720 muestras


In [22]:
print("Cargando modelo pre-entrenado...")
model_beto = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BETO,
    num_labels=len(clases),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True # Necesario ya que robertuito tiene sus propias clases originalmente
).to(device)

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=pesos_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

training_args_beto = TrainingArguments(
    output_dir='./resultados_beto',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    logging_dir='./logs_beto',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    learning_rate=LEARNING_RATE,
)

trainer_beto = CustomTrainer(
    model=model_beto,
    args=training_args_beto,
    train_dataset=train_dataset_beto,
    eval_dataset=test_dataset_beto,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Iniciando entrenamiento...")
trainer_beto.train()


[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `7`.


Cargando modelo pre-entrenado...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: pysentimiento/robertuito-emotion-analysis
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([7]) vs model:torch.Size([4])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([7, 768]) vs model:torch.Size([4, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Iniciando entrenamiento...


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,0.895750,0.744470,0.721691,0.725808,0.723901,0.722959
2,0.629791,0.744825,0.733824,0.733803,0.733274,0.733459
3,0.481033,0.778978,0.721691,0.720386,0.721146,0.720471


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1191, training_loss=0.6979952298924265, metrics={'train_runtime': 501.5604, 'train_samples_per_second': 37.952, 'train_steps_per_second': 2.375, 'total_flos': 1252102218531840.0, 'train_loss': 0.6979952298924265, 'epoch': 3.0})

In [23]:
print("Evaluando BETO en conjunto de prueba...")
beto_preds = trainer_beto.predict(test_dataset_beto)
beto_logits = beto_preds.predictions
beto_preds_ids = np.argmax(beto_logits, axis=-1)

# Métricas globales
beto_metrics = {
    'accuracy': accuracy_score(test_labels_ids, beto_preds_ids),
    'precision_macro': precision_score(test_labels_ids, beto_preds_ids, average='macro', zero_division=0),
    'recall_macro': recall_score(test_labels_ids, beto_preds_ids, average='macro', zero_division=0),
    'f1_macro': f1_score(test_labels_ids, beto_preds_ids, average='macro', zero_division=0)
}

print(f"\nResultados BETO - Test:")
print(f"   Accuracy: {beto_metrics['accuracy']:.4f}")
print(f"   Precision (macro): {beto_metrics['precision_macro']:.4f}")
print(f"   Recall (macro): {beto_metrics['recall_macro']:.4f}")
print(f"   F1-Score (macro): {beto_metrics['f1_macro']:.4f}")

# Matriz de confusión
beto_cm_path = generar_matriz_confusion(
    test_labels_ids, beto_preds_ids, clases,
    'BETO', 'cm_beto.png', RESULTADOS_DIR
)
print(f"Matriz de confusión guardada: {beto_cm_path}")

# Reporte por clase
beto_report = reporte_por_clase(test_labels_ids, beto_preds_ids, clases)

# Curva ROC
beto_probas = torch.nn.functional.softmax(torch.tensor(beto_logits), dim=-1).numpy()
beto_roc_path, beto_auc_dict, beto_macro_auc = generar_curva_roc(
    test_labels_ids, beto_probas, clases,
    'BETO', 'roc_beto.png', RESULTADOS_DIR
)
print(f"Curva ROC guardada: {beto_roc_path}")

print(f"\nAUC por clase:")
for i, clase in enumerate(clases):
    print(f"   {clase}: {beto_auc_dict[i]:.4f}")
print(f"   Macro-promedio: {beto_macro_auc:.4f}")

Evaluando BETO en conjunto de prueba...



Resultados BETO - Test:
   Accuracy: 0.7338
   Precision (macro): 0.7338
   Recall (macro): 0.7333
   F1-Score (macro): 0.7335
Matriz de confusión guardada: /content/resultados_transformers/cm_beto.png
Curva ROC guardada: /content/resultados_transformers/roc_beto.png

AUC por clase:
   Alegria: 0.8843
   Miedo: 0.9450
   Sorpresa: 0.9030
   Tristeza: 0.9049
   Macro-promedio: 0.9095


---
## 7. Entrenamiento: RoBERTa (BERTIN - Español)\n
\n
Modelo: [`bertin-project`](https://huggingface.co/xlm-roberta-base) (Español)\n
\n
Mismos hiperparámetros que BETO para comparación consistente.


In [24]:
from transformers import (
    RobertaTokenizer, RobertaForSequenceClassification,
)

In [25]:
print("Cargando tokenizer RoBERTa...")
tokenizer_roberta = RobertaTokenizer.from_pretrained('bertin-project/bertin-roberta-base-spanish')

print("Tokenizando textos de entrenamiento...")
train_encodings_roberta = tokenizer_roberta(
    train_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors=None
)

print("Tokenizando textos de prueba...")
test_encodings_roberta = tokenizer_roberta(
    test_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors=None
)

train_dataset_roberta = EmotionDataset(train_encodings_roberta, train_labels_ids)
test_dataset_roberta = EmotionDataset(test_encodings_roberta, test_labels_ids)

print(f"Train dataset: {len(train_dataset_roberta)} muestras")
print(f"Test dataset: {len(test_dataset_roberta)} muestras")


Cargando tokenizer RoBERTa...


tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/851k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/509k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.21M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Tokenizando textos de entrenamiento...
Tokenizando textos de prueba...
Train dataset: 6345 muestras
Test dataset: 2720 muestras


In [26]:
print("Cargando modelo RoBERTa...")
model_roberta = RobertaForSequenceClassification.from_pretrained(
    'bertin-project/bertin-roberta-base-spanish',
    num_labels=len(clases),
    id2label=id2label,
    label2id=label2id
)
model_roberta.to(device)

training_args_roberta = TrainingArguments(
    output_dir='/content/checkpoints_roberta',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_dir='/content/logs_roberta',
    logging_steps=50,
    report_to='none',
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    save_total_limit=1,
)

trainer_roberta = Trainer(
    model=model_roberta,
    args=training_args_roberta,
    train_dataset=train_dataset_roberta,
    eval_dataset=test_dataset_roberta,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Iniciando entrenamiento de RoBERTa...")
trainer_roberta.train()
print("Entrenamiento de RoBERTa completado.")



Cargando modelo RoBERTa...


config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: bertin-project/bertin-roberta-base-spanish
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


...

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,0.885045,0.828251,0.716912,0.723796,0.721472,0.718270
2,0.644556,0.742914,0.728676,0.731751,0.730497,0.729429
3,0.328872,0.849249,0.725000,0.725520,0.724187,0.724553


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Entrenamiento de RoBERTa completado.


In [27]:
print("Evaluando RoBERTa en conjunto de prueba...")
roberta_preds = trainer_roberta.predict(test_dataset_roberta)
roberta_logits = roberta_preds.predictions
roberta_preds_ids = np.argmax(roberta_logits, axis=-1)

# Métricas globales
roberta_metrics = {
    'accuracy': accuracy_score(test_labels_ids, roberta_preds_ids),
    'precision_macro': precision_score(test_labels_ids, roberta_preds_ids, average='macro', zero_division=0),
    'recall_macro': recall_score(test_labels_ids, roberta_preds_ids, average='macro', zero_division=0),
    'f1_macro': f1_score(test_labels_ids, roberta_preds_ids, average='macro', zero_division=0)
}

print(f"\nResultados RoBERTa - Test:")
print(f"   Accuracy: {roberta_metrics['accuracy']:.4f}")
print(f"   Precision (macro): {roberta_metrics['precision_macro']:.4f}")
print(f"   Recall (macro): {roberta_metrics['recall_macro']:.4f}")
print(f"   F1-Score (macro): {roberta_metrics['f1_macro']:.4f}")

# Matriz de confusión
roberta_cm_path = generar_matriz_confusion(
    test_labels_ids, roberta_preds_ids, clases,
    'RoBERTa', 'cm_roberta.png', RESULTADOS_DIR
)
print(f"Matriz de confusión guardada: {roberta_cm_path}")

# Reporte por clase
roberta_report = reporte_por_clase(test_labels_ids, roberta_preds_ids, clases)

# Curva ROC
roberta_probas = torch.nn.functional.softmax(torch.tensor(roberta_logits), dim=-1).numpy()
roberta_roc_path, roberta_auc_dict, roberta_macro_auc = generar_curva_roc(
    test_labels_ids, roberta_probas, clases,
    'RoBERTa', 'roc_roberta.png', RESULTADOS_DIR
)
print(f"Curva ROC guardada: {roberta_roc_path}")

print(f"\nAUC por clase:")
for i, clase in enumerate(clases):
    print(f"   {clase}: {roberta_auc_dict[i]:.4f}")
print(f"   Macro-promedio: {roberta_macro_auc:.4f}")

Evaluando RoBERTa en conjunto de prueba...



Resultados RoBERTa - Test:
   Accuracy: 0.7287
   Precision (macro): 0.7318
   Recall (macro): 0.7305
   F1-Score (macro): 0.7294
Matriz de confusión guardada: /content/resultados_transformers/cm_roberta.png
Curva ROC guardada: /content/resultados_transformers/roc_roberta.png

AUC por clase:
   Alegria: 0.8879
   Miedo: 0.9343
   Sorpresa: 0.8982
   Tristeza: 0.9002
   Macro-promedio: 0.9054


---
## 8. Generación del Reporte Unificado de Transformers\n
\n
Se guarda un solo archivo markdown con los resultados de ambos modelos para comparación directa.

In [28]:
cm_beto_rel = beto_cm_path.replace('\\', '/')
cm_roberta_rel = roberta_cm_path.replace('\\', '/')
roc_beto_rel = beto_roc_path.replace('\\', '/')
roc_roberta_rel = roberta_roc_path.replace('\\', '/')

ruta_reporte = os.path.join(RESULTADOS_DIR, 'reporte_transformers.md')

with open(ruta_reporte, 'w', encoding='utf-8') as f:
    f.write("# Reporte de Evaluación de Modelos Transformer\n\n")
    f.write("Resultados de métricas de rendimiento para los modelos BETO (BERT en Español) y RoBERTa (BERTIN - Español).\n\n")

    f.write("## Hiperparámetros\n\n")
    f.write(f"| Parámetro | Valor |\n")
    f.write(f"|-----------|-------|\n")
    f.write(f"| Max length | {MAX_LENGTH} |\n")
    f.write(f"| Batch size | {BATCH_SIZE} |\n")
    f.write(f"| Learning rate | {LEARNING_RATE} |\n")
    f.write(f"| Epochs | {EPOCHS} |\n")
    f.write(f"| Weight decay | {WEIGHT_DECAY} |\n")
    f.write(f"| Warmup steps | {WARMUP_STEPS} |\n")
    f.write(f"| Early stopping patience | 2 |\n")
    f.write(f"| Pesos de clase | balanced |\n\n")

    f.write("---\n\n")
    f.write("## Comparativa Global\n\n")
    f.write("| Métrica | BETO | RoBERTa |\n")
    f.write("|---------|------|---------|\n")
    f.write(f"| Accuracy | {beto_metrics['accuracy']:.4f} | {roberta_metrics['accuracy']:.4f} |\n")
    f.write(f"| Precision (macro) | {beto_metrics['precision_macro']:.4f} | {roberta_metrics['precision_macro']:.4f} |\n")
    f.write(f"| Recall (macro) | {beto_metrics['recall_macro']:.4f} | {roberta_metrics['recall_macro']:.4f} |\n")
    f.write(f"| F1-Score (macro) | {beto_metrics['f1_macro']:.4f} | {roberta_metrics['f1_macro']:.4f} |\n")
    f.write(f"| AUC (macro) | {beto_macro_auc:.4f} | {roberta_macro_auc:.4f} |\n\n")

    f.write("---\n\n")
    f.write("## BETO\n\n")
    f.write("### Métricas Globales\n\n")
    f.write(f"- **Accuracy:** {beto_metrics['accuracy']:.4f}\n")
    f.write(f"- **Precision (macro):** {beto_metrics['precision_macro']:.4f}\n")
    f.write(f"- **Recall (macro):** {beto_metrics['recall_macro']:.4f}\n")
    f.write(f"- **F1-Score (macro):** {beto_metrics['f1_macro']:.4f}\n\n")

    f.write("### AUC (Área Bajo la Curva ROC)\n\n")
    for i, clase in enumerate(clases):
        f.write(f"- **{clase}:** {beto_auc_dict[i]:.4f}\n")
    f.write(f"- **Macro-promedio:** {beto_macro_auc:.4f}\n\n")

    f.write("### Reporte por Clase\n\n")
    f.write("| Clase | Precisión | Recall | F1-Score | Soporte |\n")
    f.write("|-------|-----------|--------|----------|---------|\n")
    for clase in clases:
        r = beto_report[clase]
        f.write(f"| {clase} | {r['precision']:.4f} | {r['recall']:.4f} | {r['f1-score']:.4f} | {int(r['support'])} |\n")
    f.write("\n")

    f.write("### Matriz de Confusión\n\n")
    f.write(f"![Matriz de Confusión BETO]({cm_beto_rel})\n\n")

    f.write("### Curva ROC\n\n")
    f.write(f"![Curva ROC BETO]({roc_beto_rel})\n\n")

    f.write("### Classification Report\n\n")
    f.write("```text\n")
    f.write(classification_report(test_labels_ids, beto_preds_ids,
                                  labels=range(len(clases)), target_names=clases, zero_division=0))
    f.write("```\n\n")

    f.write("---\n\n")
    f.write("## RoBERTa\n\n")
    f.write("### Métricas Globales\n\n")
    f.write(f"- **Accuracy:** {roberta_metrics['accuracy']:.4f}\n")
    f.write(f"- **Precision (macro):** {roberta_metrics['precision_macro']:.4f}\n")
    f.write(f"- **Recall (macro):** {roberta_metrics['recall_macro']:.4f}\n")
    f.write(f"- **F1-Score (macro):** {roberta_metrics['f1_macro']:.4f}\n\n")

    f.write("### AUC (Área Bajo la Curva ROC)\n\n")
    for i, clase in enumerate(clases):
        f.write(f"- **{clase}:** {roberta_auc_dict[i]:.4f}\n")
    f.write(f"- **Macro-promedio:** {roberta_macro_auc:.4f}\n\n")

    f.write("### Reporte por Clase\n\n")
    f.write("| Clase | Precisión | Recall | F1-Score | Soporte |\n")
    f.write("|-------|-----------|--------|----------|---------|\n")
    for clase in clases:
        r = roberta_report[clase]
        f.write(f"| {clase} | {r['precision']:.4f} | {r['recall']:.4f} | {r['f1-score']:.4f} | {int(r['support'])} |\n")
    f.write("\n")

    f.write("### Matriz de Confusión\n\n")
    f.write(f"![Matriz de Confusión RoBERTa]({cm_roberta_rel})\n\n")

    f.write("### Curva ROC\n\n")
    f.write(f"![Curva ROC RoBERTa]({roc_roberta_rel})\n\n")

    f.write("### Classification Report\n\n")
    f.write("```text\n")
    f.write(classification_report(test_labels_ids, roberta_preds_ids,
                                  labels=range(len(clases)), target_names=clases, zero_division=0))
    f.write("```\n\n")

print(f"\nReporte unificado generado: {ruta_reporte}")
print("\n=== PROCESO COMPLETADO ===")
print("Se generaron:")
print(f"  - {os.path.basename(beto_cm_path)}")
print(f"  - {os.path.basename(beto_roc_path)}")
print(f"  - {os.path.basename(roberta_cm_path)}")
print(f"  - {os.path.basename(roberta_roc_path)}")
print(f"  - reporte_transformers.md")



Reporte unificado generado: /content/resultados_transformers/reporte_transformers.md

=== PROCESO COMPLETADO ===
Se generaron:
  - cm_beto.png
  - roc_beto.png
  - cm_roberta.png
  - roc_roberta.png
  - reporte_transformers.md
